# 1) Carga de Librerias

In [1]:
import pandas as pd
import numpy as np
from LLM import Clasificador
from tqdm import tqdm

e:\TuttiQuanti\Trabajo\Anaconda3\envs\env_LLMzCor\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2) Carga de BD de Tratamientos (Opcional)

In [2]:
#Caraga la base de datos de los tratamientos (tx) para cada bacteria
treatment_path = r"E:\PaperLLM\LLMzCor.github.io\DBs\Treatment_2017.xlsx"
tx_df = pd.read_excel(io= treatment_path, index_col=0)
#tx_df.head()


In [3]:
#Selecciona los tratamiento de primera eleccion utilizados hasta 2017
Bacteria = "Chlamydia_trachomatis"
treatment= tx_df.loc[Bacteria,"First-line treatment until 2017"]
print(treatment)

Azithromycin 1 g PO single dose or doxycycline 100 mg PO every 12 hours for 7 days.


# 3) Carga BD de Bacterias

In [4]:
file_path = r"E:\PaperLLM\LLMzCor.github.io\DBs\Chlamydia_trachomatis.xlsx"
sheet = "2012-2022"
usecols= [
    'PMID',
    'Title',
    'Abstract',
    'Estado',
    '1) Antimicrobial Resistance stain',
    '2) New treatment',
    '3) Immunization',
    'Human_summary',
    'Publication Year',
    'Journal/Book',
    'Alerta'
]
df_unfilter = pd.read_excel(
    io=file_path,
    sheet_name=sheet,
    usecols=usecols, index_col=0
)

In [5]:
#df-unfilter.head()
df_unfilter.shape

(4810, 10)

In [6]:
#Filtro los excluidos
df= df_unfilter[df_unfilter['Estado']!="Excluir"]

df.shape

(4792, 10)

In [5]:
#Calculo cuantos paper se excluyeron
resta = df_unfilter.shape[0]- df.shape[0]
print(resta)

18


In [6]:
df.head()

,Title,Journal/Book,Publication Year,Alerta,Estado,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary
PMID,,,,,,,,,,
22304240,O(2)-evolving chlorite dismutase as a tool for...,Biochemistry,2012,PreAlerta,Revisar,NaN,NaN,NaN,NaN,NaN
21848430,Differing effects of azithromycin and doxycycl...,DNA Cell Biol,2012,PreAlerta,Revisar,NaN,NaN,NaN,NaN,NaN
22042092,Absence of Swedish new variant Chlamydia trach...,Acta Derm Venereol,2012,PreAlerta,Listo,NaN,NaN,NaN,NaN,NaN
22203235,Chlamydia pneumoniae entry into epithelial cel...,Microb Pathog,2012,PreAlerta,Listo,NaN,NaN,NaN,NaN,NaN
22222354,High-valent [MnFe] and [FeFe] cofactors in rib...,Biochim Biophys Acta,2012,PreAlerta,Revisar,NaN,NaN,NaN,NaN,NaN


# 4) LLM

In [7]:
clasificador=Clasificador()

In [8]:
def _transformacion_binario_val(col):
    if (col=="Yes") | (col==1):
        return 1
    elif (col=="No") | (col==0):
        return 0
def _transformacion_binario_df(df):
    df[['1) Antimicrobial Resistance stain',
        '2) New treatment','3) Immunization']] = df[['1) Antimicrobial Resistance stain',
                                                     '2) New treatment',
                                                     '3) Immunization']].applymap(_transformacion_binario_val)
    return df



def ask_llm(df_,partition=0):
    df=df_.copy()
    df=_transformacion_binario_df(df)
    df["ai_label"]=np.nan
    df["ai_summary"]=np.nan
    if partition==0:
        for pmid in df.index:
            try:
                response = clasificador.clasificacion(df.loc[pmid,"Title"])
                df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
            except Exception as e:
                print(f"Se corto el proceso del LLM por este motivo : {e} en el id {pmid}")
                return df.loc[:pmid].iloc[:-1]
        return df
    else:
        print("aca")
        df_ptit=df.iloc[:partition,:]
        for pmid in df_ptit.index:
            try:
                response = clasificador.clasificacion(df_ptit.loc[pmid,"Title"])
                df_ptit.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
            except Exception as e:
                print(f"Se corto el proceso del LLM por este motivo : {e} en el id {pmid}")
                return df_ptit.loc[:pmid].iloc[:-1]
        return df_ptit

def _evaluate(row):
    values = row[["1) Antimicrobial Resistance stain","2) New treatment","3) Immunization"]].values
    ai_opcion=int(row["ai_label"])-1
    print(f"values={values}")
    print(f"ai_opcion{ai_opcion}")
    if (values.sum()==0) & (ai_opcion==3):
        return 1
    elif (values.sum()==0) & (ai_opcion<3):
        return 0
    elif (values.sum()>0) & (ai_opcion==3):
        return 0
    elif values[ai_opcion]>0:

        return 1
    else:
        return 0

def evaluacion_score(df,partition=0):
    if partition ==0:
        n=df.shape[0]
        scores = df.apply(_evaluate,axis=1)
        final_score=sum(scores)/n
    else:
        df_ptit=df.iloc[:10,:]
        n=df_ptit.shape[0]
        scores=df_ptit.apply(_evaluate,axis=1)
        final_score=sum(scores)/n
    return final_score

### Prueba un paper (Opcional)

In [56]:

#paper=df.loc[22304240, "Abstract"]
paper=df.loc[22525317, "Title"]
clasificador.clasificacion(paper)

['2',
 " 'The paper discusses derivatives of 8-hydroxyquinoline, which are antibacterial agents targeting intra- and extracellular Gram-negative pathogens, implying new treatment methods.'"]

# 5) Ask LLM 

---
Abreviatura dataframes de bacterias:
+ df_Ct --> Chlamydia trachomatis
+ df_Cd --> Clostridium difficile
+ df_Hi --> Haemopilus influenzae
+ df_Kp --> Klebsiella pneumoniae
+ df_Ng --> Neisseria gonorrhoeae
+ df_Sh --> Shigela spp
---

### Prueba partición del df (Opcional)

In [9]:
# df_ptit=df.iloc[:10,:]
# shape_df_ptit= df_ptit.shape
# df_Ct_classified_1 = ask_llm(df_ptit)
# shape_df= df_Ct_classified_1.shape


# print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

In [15]:
#df_Ct_classified_1

### 📝 Para aplicar a la BDs completa necesitamos los abstracts... Podremos hacer un Web scraping 🤔❓ 

En las BDs completa las columnas estarán vacias, por lo tanto solo tendremos las ai_label y ai_summary

+ Antimicrobial Resistance stain
+ New treatment
+ Immunization
+ Abstract 
+ Human_summary

In [10]:

columns_excluded = ['1) Antimicrobial Resistance stain', 
                    '2) New treatment', 
                    '3) Immunization', 
                    'Abstract', 
                    'Human_summary'
                    ]

In [ ]:
Ct_df1= df_Ct_classified_1.drop(columns= columns_excluded)

,Title,Journal/Book,Publication Year,Alerta,Estado,ai_label,ai_summary
PMID,,,,,,,
22304240,O(2)-evolving chlorite dismutase as a tool for...,Biochemistry,2012,PreAlerta,Revisar,4,'The paper does not discuss Multiresistance b...
21848430,Differing effects of azithromycin and doxycycl...,DNA Cell Biol,2012,PreAlerta,Revisar,2,'The paper discusses the effects of two antib...
22042092,Absence of Swedish new variant Chlamydia trach...,Acta Derm Venereol,2012,PreAlerta,Listo,4,'The paper does not discuss Multiresistance b...
22203235,Chlamydia pneumoniae entry into epithelial cel...,Microb Pathog,2012,PreAlerta,Listo,4,'The paper discusses the entry of Chlamydia p...
22222354,High-valent [MnFe] and [FeFe] cofactors in rib...,Biochim Biophys Acta,2012,PreAlerta,Revisar,4,'The abstract discusses high-valent [MnFe
22337116,Stable Chlamydia prevalence does not exclude i...,Sex Transm Dis,2012,PreAlerta,NaN,4,"""The abstract does not discuss multiresistanc..."
22369307,The use of polymerase chain reaction assay ver...,Arch Iran Med,2012,PreAlerta,NaN,4,'The paper does not discuss Multiresistance b...
22382226,Frequency of Chlamydia trachomatis among male ...,Saudi J Kidney Dis Transpl,2012,PreAlerta,NaN,4,'The paper does not discuss multiresistance b...
22386127,"Serum interleukin-1β, interleukin-8 and anti-h...",J Reprod Immunol,2012,PreAlerta,NaN,4,'The paper does not discuss Multiresistance b...


# 6) Ask_LLMzCor dfs

In [11]:
df_ptit=df.iloc[:10,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_1 = ask_llm(df_ptit)
shape_df= df_Ct_classified_1.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper does not discuss Multiresistance bacteria stains report, neither new treatments nor immunization. Instead, it focuses on using O(2)-evolving chlorite dismutase as a tool for studying O(2)-utilizing enzymes, whic

Shape df_ptit  (10, 10) , Shape df_classified  (10, 12)


In [12]:
df_ptit=df.iloc[10:20,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_2 = ask_llm(df_ptit)
shape_df= df_Ct_classified_2.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

Se corto el proceso del LLM por este motivo : list index out of range en el id 22422335
Shape df_ptit  (10, 10) , Shape df_classified  (0, 12)


C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)


In [20]:
df_ptit=df.iloc[20:30,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_3 = ask_llm(df_ptit)
shape_df= df_Ct_classified_3.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The abstract does not discuss multiresistance bacteria stains report, new treatments, or immunization. Instead, it focuses on inflammatory pathways in cervical cancer, which is not covered by the provided categories."' ha

Se corto el proceso del LLM por este motivo : list index out of range en el id 21938501
Shape df_ptit  (10, 10) , Shape df_classified  (3, 12)


In [15]:
df_Ct_classified_3

,Title,Journal/Book,Publication Year,Alerta,Estado,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,,,,,
22668947,Inflammatory pathways in cervical cancer - the...,S Afr Med J,2012,Excluido,NaN,NaN,NaN,NaN,NaN,NaN,4,"""The abstract does not discuss multiresistanc..."
22341824,Prevention of sexually transmitted infections ...,Lancet,2012,Excluido,NaN,NaN,NaN,NaN,NaN,NaN,4,'The abstract does not discuss Multiresistanc...
22456736,Genome-wide recombination in Chlamydia trachom...,Nat Genet,2012,PreAlerta,NaN,NaN,NaN,NaN,NaN,NaN,4,'The abstract does not discuss multiresistanc...


In [18]:
df_ptit=df.iloc[30:40,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_4 = ask_llm(df_ptit)
shape_df= df_Ct_classified_4.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper does not discuss Multiresistance bacteria stains report, neither new treatments nor immunization. Instead, it focuses on the correlation of Atopobium vaginae amount with Bacterial Vaginosis markers, which falls 

Se corto el proceso del LLM por este motivo : list index out of range en el id 22384841
Shape df_ptit  (10, 10) , Shape df_classified  (1, 12)


In [19]:
df_ptit=df.iloc[40:50,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_5 = ask_llm(df_ptit)
shape_df= df_Ct_classified_5.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The abstract does not discuss Multiresistance bacteria stains report, new treatments, nor immunization. It focuses on Pelvic inflammatory disease, which does not fall into the provided categories.'' has dtype incompatible

Se corto el proceso del LLM por este motivo : list index out of range en el id 22387629
Shape df_ptit  (10, 10) , Shape df_classified  (1, 12)


In [21]:
df_ptit=df.iloc[50:60,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_6 = ask_llm(df_ptit)
shape_df= df_Ct_classified_6.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)


Se corto el proceso del LLM por este motivo : list index out of range en el id 22293657
Shape df_ptit  (10, 10) , Shape df_classified  (0, 12)


In [22]:
df_ptit=df.iloc[60:70,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_7 = ask_llm(df_ptit)
shape_df= df_Ct_classified_7.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The abstract does not discuss multiresistance bacteria stains report, new treatments, or immunization. Instead, it focuses on the sensitivity of 20-minute voiding intervals in men testing for Chlamydia trachomatis, which 

Se corto el proceso del LLM por este motivo : list index out of range en el id 22563554
Shape df_ptit  (10, 10) , Shape df_classified  (2, 12)


In [37]:
df_ptit=df.iloc[70:80,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_8 = ask_llm(df_ptit)
shape_df= df_Ct_classified_8.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The abstract does not discuss multiresistance bacteria stains report, new treatments, or immunization. Instead, it seems to be about identifying biomarkers for trachomatous trichiasis, a condition that can lead to blindne

Se corto el proceso del LLM por este motivo : list index out of range en el id 22251247
Shape df_ptit  (10, 10) , Shape df_classified  (7, 12)


In [24]:
df_ptit=df.iloc[80:90,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_9 = ask_llm(df_ptit)
shape_df= df_Ct_classified_9.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The paper discusses behavioral and sociodemographic risk factors for HPV6, 11, 16, 18 infections, which does not align with the descriptions of multiresistance bacteria stains report, new treatments, or immunization."' ha

Se corto el proceso del LLM por este motivo : list index out of range en el id 22308534
Shape df_ptit  (10, 10) , Shape df_classified  (2, 12)


In [25]:
df_ptit=df.iloc[90:100,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_10 = ask_llm(df_ptit)
shape_df= df_Ct_classified_10.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The abstract does not discuss multiresistance bacteria stains report, new treatments, nor immunization. Instead, it focuses on the first performance report for the Bio-Rad Dx CT/NG/MG assay, which is a diagnostic tool for

Shape df_ptit  (10, 10) , Shape df_classified  (10, 12)


In [26]:
df_ptit=df.iloc[100:110,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_11 = ask_llm(df_ptit)
shape_df= df_Ct_classified_11.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The abstract of the paper does not discuss multiresistance bacteria stains report, new treatments, or immunization. Instead, it focuses on risk factors for male patients with gonorrhoea complicated by inflammation of the 

Se corto el proceso del LLM por este motivo : list index out of range en el id 22816103
Shape df_ptit  (10, 10) , Shape df_classified  (2, 12)


In [27]:
df_ptit=df.iloc[110:120,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_12 = ask_llm(df_ptit)
shape_df= df_Ct_classified_12.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The paper discusses the identification of Chlamydia trachomatis antigens that can elicit T cell and antibody responses, which aligns with the description of category 3, Immunization."' has dtype incompatible with float64,

Se corto el proceso del LLM por este motivo : list index out of range en el id 22421110
Shape df_ptit  (10, 10) , Shape df_classified  (8, 12)


In [28]:
df_ptit=df.iloc[120:130,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_13 = ask_llm(df_ptit)
shape_df= df_Ct_classified_13.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper discusses the development of engineered phage-based therapeutic materials, which can be classified as new treatments as they aim to inhibit Chlamydia trachomatis intracellular infection.'' has dtype incompatible

Se corto el proceso del LLM por este motivo : list index out of range en el id 22665342
Shape df_ptit  (10, 10) , Shape df_classified  (8, 12)


In [30]:
df_ptit=df.iloc[130:140,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_14 = ask_llm(df_ptit)
shape_df= df_Ct_classified_14.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The abstract does not discuss multiresistance bacteria stains report, new treatments, or immunization. Instead, it focuses on the relationship between subclinical pelvic inflammatory disease and infertility.'' has dtype i

Shape df_ptit  (10, 10) , Shape df_classified  (10, 12)


In [31]:
df_ptit=df.iloc[140:150,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_15 = ask_llm(df_ptit)
shape_df= df_Ct_classified_15.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The abstract of the paper does not discuss Multiresistance bacteria stains report, new treatments, nor immunization. Instead, it focuses on the impact of loci nature on estimating recombination and mutation rates in Chlam

Se corto el proceso del LLM por este motivo : list index out of range en el id 22937701
Shape df_ptit  (10, 10) , Shape df_classified  (3, 12)


In [32]:
df_ptit=df.iloc[150:160,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_16 = ask_llm(df_ptit)
shape_df= df_Ct_classified_16.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper does not discuss Multiresistance bacteria stains report, neither new treatments nor immunization. Instead, it focuses on the effectiveness of yearly, register based screening for chlamydia in the Netherlands.'' 

Shape df_ptit  (10, 10) , Shape df_classified  (10, 12)


In [33]:
df_ptit=df.iloc[160:170,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_17 = ask_llm(df_ptit)
shape_df= df_Ct_classified_17.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper does not discuss Multiresistance bacteria stains report, neither new treatments nor immunization. Instead, it focuses on the comparative effectiveness of two self-collected sample kit distribution systems for ch

Shape df_ptit  (10, 10) , Shape df_classified  (10, 12)


In [34]:
df_ptit=df.iloc[170:180,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_18 = ask_llm(df_ptit)
shape_df= df_Ct_classified_18.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' "The paper does not discuss multiresistance bacteria stains report, new treatments, or immunization. Instead, it focuses on estimating the proportion of tubal factor infertility caused by Chlamydia, using serological evide

Shape df_ptit  (10, 10) , Shape df_classified  (10, 12)


In [35]:
df_ptit=df.iloc[180:190,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_19 = ask_llm(df_ptit)
shape_df= df_Ct_classified_19.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)


Se corto el proceso del LLM por este motivo : list index out of range en el id 23007214
Shape df_ptit  (10, 10) , Shape df_classified  (0, 12)


In [36]:
df_ptit=df.iloc[190:200,:]
shape_df_ptit= df_ptit.shape

df_Ct_classified_20 = ask_llm(df_ptit)
shape_df= df_Ct_classified_20.shape


print("Shape df_ptit ", shape_df_ptit, ", Shape df_classified ", shape_df )

C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '4' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
C:\Users\Windows 11\AppData\Local\Temp\ipykernel_8544\3859912656.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value ' 'The paper does not discuss multiresistance bacteria stains report, new treatments, nor immunization. Instead, it focuses on the correlation of lifestyle and medical conditions with vaginal infections and the production of

Se corto el proceso del LLM por este motivo : list index out of range en el id 22795612
Shape df_ptit  (10, 10) , Shape df_classified  (1, 12)


In [ ]:
df_list_Ct= [df_Ct_classified_1, 
             df_Ct_classified_2, 
             df_Ct_classified_3, 
             df_Ct_classified_4, 
             df_Ct_classified_5, 
             df_Ct_classified_6, 
             df_Ct_classified_8, 
             df_Ct_classified_7, 
             df_Ct_classified_9, 
             df_Ct_classified_10, 
             df_Ct_classified_11, 
             df_Ct_classified_12,
             df_Ct_classified_13, 
             df_Ct_classified_14, 
             df_Ct_classified_15, 
             df_Ct_classified_16,
             df_Ct_classified_17]
df_Cta = pd.concat(df_list_Ct, ignore_index=False)
df_Ct_c.shape